# LeetCode 525 — Contiguous Array bằng Python

Link đề: https://leetcode.com/problems/contiguous-array/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def findMaxLength(self, nums: List[int]) -> int:
        ...
```

Ghi chú: bài này chỉ dùng list và dict của Python, không cần cấu trúc tự dựng. Toàn bộ đáp án trong notebook đã được kiểm chứng bằng code (đối chiếu với cách vét cạn độc lập).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong mảng nhị phân (chỉ có `0` và `1`), tìm **đoạn con liên tiếp dài nhất** có **số `0` bằng số `1`**, và trả về **độ dài** của đoạn đó.
2. Nắm các công cụ Python cần thiết: cắt lát `nums[i:j]`, `list.count`, `dict` (tạo, tra cứu, `in`, `get`), `enumerate`, `itertools.accumulate`, và mẹo biến `0` thành `-1` bằng biểu thức `2 * x - 1`.
3. Đi từ **vét cạn mọi đoạn con `O(n^2)`** → **tổng tiền tố (prefix sum)** → **hash map lưu vị trí xuất hiện đầu tiên `O(n)`** → **mảng thay cho dict** → **gom nhóm theo giá trị tiền tố, lấy vị trí đầu và cuối**.
4. Hiểu mấu chốt: "số `0` bằng số `1`" tương đương "**tổng bằng 0** khi coi `0` là `-1`", và một đoạn có tổng 0 chính là cặp hai tiền tố **bằng nhau**.
5. Tránh các lỗi kinh điển: quên "gốc" `{0: -1}`, ghi đè vị trí cũ khi gặp lại tiền tố, đếm `0` và `1` rời nhau rồi so sánh toàn mảng, và nhầm độ dài `i - j` với `i - j + 1`.

## 2. Đề bài & ví dụ

Cho mảng nhị phân `nums` (mỗi phần tử là `0` hoặc `1`). Hãy trả về **độ dài lớn nhất** của một **đoạn con liên tiếp** (contiguous subarray) có **số lượng `0` bằng số lượng `1`**.

**Ví dụ 1**

- Input: `nums = [0,1]`
- Output: `2`
- Giải thích: `[0, 1]` là đoạn dài nhất có số `0` và `1` bằng nhau.

**Ví dụ 2**

- Input: `nums = [0,1,0]`
- Output: `2`
- Giải thích: `[0, 1]` (hoặc `[1, 0]`) là đoạn dài nhất cân bằng.

**Ví dụ 3**

- Input: `nums = [0,1,1,1,1,1,0,0,0]`
- Output: `6`
- Giải thích: `[1,1,1,0,0,0]` là đoạn dài nhất có số `0` bằng số `1`.

**Ràng buộc:**

- `1 <= nums.length <= 10^5`
- `nums[i]` chỉ là `0` hoặc `1`.

Chú ý: nếu **không có** đoạn nào cân bằng (ví dụ `[0,0,0]`) thì đáp án là `0`. Với `n` đến `10^5`, cách thử mọi đoạn `O(n^2)` (khoảng `5 * 10^9` bước) sẽ quá chậm.

In [1]:
from typing import List
from itertools import accumulate
import random
import time

examples = [
    ([0, 1], 2),
    ([0, 1, 0], 2),
    ([0, 1, 1, 1, 1, 1, 0, 0, 0], 6),
]
for nums, expected in examples:
    print(nums, "-> kỳ vọng", expected)

[0, 1] -> kỳ vọng 2
[0, 1, 0] -> kỳ vọng 2
[0, 1, 1, 1, 1, 1, 0, 0, 0] -> kỳ vọng 6


## 3. Kiến thức nền

Trước khi giải, ta làm quen với vài công cụ Python. Mỗi công cụ đi kèm một ví dụ nhỏ để bạn tự chạy và quan sát.

### 3.1. Đoạn con liên tiếp và cắt lát `nums[i:j]`

**Đoạn con liên tiếp (subarray)** là một dãy các phần tử đứng **sát nhau** trong mảng, khác với *subsequence* (dãy con) có thể bỏ cách quãng. Trong Python, `nums[i:j]` lấy các phần tử có chỉ số `i, i+1, ..., j-1` (**lấy `i`, không lấy `j`**). Độ dài của lát cắt là `j - i`.

In [2]:
nums = [0, 1, 1, 0, 1]
print(nums[0:2])        # chỉ số 0 và 1
print(nums[1:4])        # chỉ số 1, 2, 3
print(nums[2:2])        # rỗng, vì j - i = 0
print(len(nums[1:4]))   # 3 = 4 - 1

[0, 1]
[1, 1, 0]
[]
3


### 3.2. Đếm phần tử bằng `list.count`

`lst.count(x)` trả về số lần `x` xuất hiện trong `lst`. Ta dùng nó để kiểm tra một đoạn có "cân bằng" hay không: số `0` bằng số `1`. Cách này chậm (mỗi lần đếm mất `O(độ dài)`), nhưng cực dễ hiểu nên sẽ làm **oracle** (đáp án kiểm chứng) ở phần test.

In [3]:
seg = [0, 1, 1, 0, 1, 1]
print("số 0:", seg.count(0), "| số 1:", seg.count(1))
print("cân bằng?", seg.count(0) == seg.count(1))
print("cân bằng? (đoạn đầu 4 phần tử)", seg[:4].count(0) == seg[:4].count(1))

số 0: 2 | số 1: 4
cân bằng? False
cân bằng? (đoạn đầu 4 phần tử) True


### 3.3. `dict`: tra cứu nhanh theo khóa

`dict` (từ điển) lưu các cặp **khóa → giá trị**. Tra cứu, thêm và kiểm tra tồn tại đều `O(1)` trung bình.

- `d = {}` tạo dict rỗng, `d[k] = v` thêm/ghi đè.
- `k in d` kiểm tra khóa có trong dict hay không (**không** kiểm tra giá trị).
- `d.get(k, mặc_định)` lấy giá trị, nếu thiếu khóa thì trả về giá trị mặc định thay vì báo lỗi `KeyError`.

Khóa có thể là **số âm** hoặc `0`, hoàn toàn bình thường. Điều này quan trọng vì tổng tiền tố của ta sẽ có lúc âm.

In [4]:
d = {0: -1}                 # khóa 0 -> giá trị -1
d[-2] = 5                   # khóa âm vẫn ổn
print(d)
print(-2 in d, 7 in d)      # True False
print(d.get(7, "không có")) # không có
print(d.get(0))             # -1
d[-2] = 9                   # ghi đè giá trị cũ
print(d)

{0: -1, -2: 5}
True False
không có
-1
{0: -1, -2: 9}


### 3.4. `enumerate` và tổng tiền tố bằng `accumulate`

`enumerate(lst)` cho ta cả **chỉ số** lẫn **giá trị** khi duyệt: `for i, x in enumerate(lst)`.

**Tổng tiền tố (prefix sum)**: với dãy `a`, tiền tố thứ `k` là `a[0] + a[1] + ... + a[k-1]` (tổng `k` phần tử đầu). Trong Python, `itertools.accumulate(a)` sinh các tổng cộng dồn. Ta quy ước thêm tiền tố `0` ở đầu (tổng của **không** phần tử nào).

In [5]:
a = [3, -1, 4, -4]
for i, x in enumerate(a):
    print("chỉ số", i, "giá trị", x)

pref = [0] + list(accumulate(a))
print("tiền tố:", pref)           # [0, 3, 2, 6, 2]
# Tổng đoạn a[1..3] (chỉ số 1,2,3) = pref[4] - pref[1]
print("tổng a[1:4] =", sum(a[1:4]), "==", pref[4] - pref[1])

chỉ số 0 giá trị 3
chỉ số 1 giá trị -1
chỉ số 2 giá trị 4
chỉ số 3 giá trị -4
tiền tố: [0, 3, 2, 6, 2]
tổng a[1:4] = -1 == -1


Công thức quan trọng: **tổng đoạn `a[j:i]` = `pref[i] - pref[j]`** (đoạn gồm các chỉ số `j, j+1, ..., i-1`). Nhờ nó, ta không phải cộng lại từng đoạn nữa.

### 3.5. Biến `0` thành `-1` bằng `2 * x - 1`

Với `x` chỉ là `0` hoặc `1`: nếu `x = 1` thì `2 * 1 - 1 = 1`; nếu `x = 0` thì `2 * 0 - 1 = -1`. Vậy phép biến đổi này đổi **`0` thành `-1`** và **giữ nguyên `1`**. Sau khi đổi, "số `1` bằng số `0`" nghĩa là các số `+1` và `-1` **triệt tiêu nhau**, tức tổng bằng `0`.

In [6]:
nums = [0, 1, 1, 0, 0, 1]
signed = [2 * x - 1 for x in nums]
print(nums)
print(signed)
print("tổng cả mảng:", sum(signed))
print("đoạn [1,1,0,0] có tổng:", sum(signed[1:5]))

[0, 1, 1, 0, 0, 1]
[-1, 1, 1, -1, -1, 1]
tổng cả mảng: 0
đoạn [1,1,0,0] có tổng: 0


## 4. Ý tưởng nền tảng / trực giác

Ta cần đoạn có số `0` bằng số `1`. Hãy tưởng tượng bạn đi dọc mảng và giữ một con số **"cân bằng"** `bal`:

- gặp `1`: `bal` **tăng 1**,
- gặp `0`: `bal` **giảm 1**.

Nếu từ vị trí `j` đến vị trí `i` mà `bal` **trở về đúng giá trị cũ**, nghĩa là trong đoạn đó mức tăng và mức giảm **bù trừ hoàn toàn** — số `1` bằng số `0`.

Tính tay với ví dụ 3: `nums = [0,1,1,1,1,1,0,0,0]`.

| Sau khi đọc | phần tử | `bal` |
|---|---|---|
| (bắt đầu) | | 0 |
| chỉ số 0 | 0 | -1 |
| chỉ số 1 | 1 | 0 |
| chỉ số 2 | 1 | 1 |
| chỉ số 3 | 1 | 2 |
| chỉ số 4 | 1 | 3 |
| chỉ số 5 | 1 | 4 |
| chỉ số 6 | 0 | 3 |
| chỉ số 7 | 0 | 2 |
| chỉ số 8 | 0 | 1 |

`bal = 1` xuất hiện **lần đầu** sau chỉ số 2 và **lần cuối** sau chỉ số 8. Đoạn giữa hai thời điểm đó là các chỉ số `3..8`, độ dài `8 - 2 = 6`. Đó chính là đáp án `6`.

In [7]:
nums = [0, 1, 1, 1, 1, 1, 0, 0, 0]
bal = 0
history = [("bắt đầu", None, 0)]
for i, x in enumerate(nums):
    bal += 1 if x == 1 else -1
    history.append((f"chỉ số {i}", x, bal))

for row in history:
    print(row)

('bắt đầu', None, 0)
('chỉ số 0', 0, -1)
('chỉ số 1', 1, 0)
('chỉ số 2', 1, 1)
('chỉ số 3', 1, 2)
('chỉ số 4', 1, 3)
('chỉ số 5', 1, 4)
('chỉ số 6', 0, 3)
('chỉ số 7', 0, 2)
('chỉ số 8', 0, 1)


Nhìn dãy `bal`: `0, -1, 0, 1, 2, 3, 4, 3, 2, 1`. Các giá trị lặp lại là `0` (ở thời điểm 0 và 2), `1` (thời điểm 3 và 9), `2`, `3`. Mỗi lần lặp là một đoạn cân bằng; ta chọn cặp **xa nhau nhất**. Cặp `bal = 1` ở thời điểm 3 và 9 cách nhau `9 - 3 = 6`.

Cách đánh thời điểm: thời điểm `t` nghĩa là "sau khi đọc `t` phần tử đầu", nên thời điểm `0` là trước khi đọc gì, và `bal` khi đó là `0`.

## 5. Cách 1: Brute Force (thử mọi đoạn)

**Ý tưởng:** thử mọi cặp (điểm đầu `i`, điểm cuối `j`). Với mỗi điểm đầu `i`, ta cho điểm cuối `j` đi tới và duy trì hai biến đếm `zeros`, `ones`. Mỗi khi `zeros == ones`, ghi nhận độ dài `j - i + 1`.

Đây là cách "đếm cộng dồn" đã tránh được việc đếm lại từ đầu mỗi đoạn, nhưng vẫn có hai vòng lặp lồng nhau.

In [8]:
def brute_force(nums):
    n = len(nums)
    best = 0
    for i in range(n):              # điểm đầu
        zeros = ones = 0
        for j in range(i, n):       # điểm cuối (tính cả j)
            if nums[j] == 0:
                zeros += 1
            else:
                ones += 1
            if zeros == ones:
                best = max(best, j - i + 1)
    return best

for nums, expected in examples:
    got = brute_force(nums)
    print(nums, "->", got, "| kỳ vọng", expected)
    assert got == expected
print("không có đoạn cân bằng:", brute_force([0, 0, 0]))

[0, 1] -> 2 | kỳ vọng 2
[0, 1, 0] -> 2 | kỳ vọng 2
[0, 1, 1, 1, 1, 1, 0, 0, 0] -> 6 | kỳ vọng 6
không có đoạn cân bằng: 0


**Độ phức tạp:** hai vòng lặp lồng nhau, số cặp `(i, j)` khoảng `n(n+1)/2`, nên **Time `O(n^2)`**, **Space `O(1)`**.

**Nút thắt:** với `n = 10^5` có khoảng `5 * 10^9` cặp, quá chậm. Hãy đo thử để thấy tốc độ tăng theo bình phương.

In [9]:
rng = random.Random(0)
for n in (500, 1000, 2000, 4000):
    arr = [rng.randint(0, 1) for _ in range(n)]
    t0 = time.perf_counter()
    brute_force(arr)
    dt = time.perf_counter() - t0
    print(f"n = {n:5d}: {dt * 1000:8.1f} ms")
print("n tăng gấp đôi thì thời gian tăng gần gấp bốn -> O(n^2)")

n =   500:     10.3 ms
n =  1000:     41.9 ms


n =  2000:    119.9 ms


n =  4000:    505.2 ms
n tăng gấp đôi thì thời gian tăng gần gấp bốn -> O(n^2)


Điều lãng phí: khi `j` tăng thêm 1 ta chỉ cần biết "`bal` có quay về giá trị mà ta từng thấy ở đâu đó phía trước không". Bài toán thật ra là **tìm hai thời điểm có `bal` bằng nhau và cách xa nhau nhất**.

## 6. Ý tưởng thuật toán / bước đột phá

Gọi `pref[t]` là `bal` sau khi đọc `t` phần tử (`pref[0] = 0`), với quy tắc `1 -> +1`, `0 -> -1`. Khi đó:

- tổng đoạn `nums[j:i]` (chỉ số `j .. i-1`) theo quy tắc trên là `pref[i] - pref[j]`;
- đoạn đó cân bằng **khi và chỉ khi** `pref[i] - pref[j] = 0`, tức `pref[i] == pref[j]`;
- độ dài đoạn là `i - j`.

Bài toán trở thành: **trong dãy `pref`, tìm hai vị trí `j < i` có cùng giá trị sao cho `i - j` lớn nhất.**

Với mỗi giá trị `v`, ta muốn `j` **nhỏ nhất** (vị trí xuất hiện **đầu tiên** của `v`). Vì vậy ta dùng một dict `first` lưu `first[v] = vị trí đầu tiên pref nhận giá trị v`. Duyệt `i` từ trái sang phải: nếu `pref[i]` đã có trong `first` thì ứng viên là `i - first[pref[i]]`; nếu chưa thì ghi `first[pref[i]] = i`.

Hai chi tiết sống còn:

1. **Khởi tạo `first = {0: 0}`** theo thời điểm (hoặc `{0: -1}` khi dùng chỉ số mảng, vì đọc xong phần tử chỉ số `i` là thời điểm `i + 1`). Nhờ đó đoạn bắt đầu từ đầu mảng cũng được tính.
2. **Không ghi đè** `first[v]` khi đã có, vì ta muốn điểm đầu xa nhất.

In [10]:
nums = [0, 1, 1, 1, 1, 1, 0, 0, 0]
pref = [0]
for x in nums:
    pref.append(pref[-1] + (1 if x else -1))
print("pref :", pref)

# cặp (j, i) có pref bằng nhau và cách xa nhất
best = (0, 0)
for i in range(len(pref)):
    for j in range(i):
        if pref[i] == pref[j] and i - j > best[1] - best[0]:
            best = (j, i)
print("cặp xa nhất:", best, "độ dài", best[1] - best[0])
print("đoạn tương ứng:", nums[best[0]:best[1]])

pref : [0, -1, 0, 1, 2, 3, 4, 3, 2, 1]
cặp xa nhất: (3, 9) độ dài 6
đoạn tương ứng: [1, 1, 1, 0, 0, 0]


## 7. Các cách tiếp theo

Tổng cộng ta có **5 cách**: Cách 1 (đã làm) vét cạn; Cách 2 tổng tiền tố `O(n^2)`; Cách 3 hash map vị trí đầu tiên `O(n)` (cách chính); Cách 4 mảng offset thay dict; Cách 5 gom nhóm theo giá trị tiền tố lấy vị trí đầu và cuối.

### 7.1. Cách 2: Tổng tiền tố, hai vòng lặp

**Giải thích từng đoạn:**

1. Dựng mảng `pref` có `n + 1` phần tử với `pref[0] = 0`.
2. Với mỗi cặp `j < i`, nếu `pref[i] == pref[j]` thì đoạn `nums[j:i]` cân bằng, độ dài `i - j`.

Ta chỉ thay phần "đếm 0 và 1" bằng phép so sánh hai số, đã tiến gần ý tưởng cuối nhưng vẫn là hai vòng lặp.

In [11]:
def prefix_pairs(nums):
    pref = [0]
    for x in nums:
        pref.append(pref[-1] + (1 if x else -1))
    best = 0
    n = len(pref)
    for j in range(n):
        for i in range(j + 1, n):
            if pref[i] == pref[j]:
                best = max(best, i - j)
    return best

for nums, expected in examples:
    assert prefix_pairs(nums) == expected
print("Cách 2 chạy đúng 3 ví dụ")

Cách 2 chạy đúng 3 ví dụ


**Code nộp LeetCode (Cách 2, chỉ để học — sẽ bị quá thời gian với `n` lớn):**

```python
class Solution:
    def findMaxLength(self, nums: List[int]) -> int:
        pref = [0]
        for x in nums:
            pref.append(pref[-1] + (1 if x else -1))
        best, m = 0, len(pref)
        for j in range(m):
            for i in range(j + 1, m):
                if pref[i] == pref[j]:
                    best = max(best, i - j)
        return best
```

**Độ phức tạp:** Time `O(n^2)`, Space `O(n)` cho `pref`. Còn tệ hơn Cách 1 về bộ nhớ, nhưng là bước đệm để hiểu cặp "tiền tố bằng nhau".

### 7.2. Cách 3: Hash map lưu vị trí đầu tiên (cách chính)

**Giải thích từng đoạn:**

1. `first = {0: -1}`: tiền tố rỗng (trước phần tử chỉ số 0) có tổng `0`, coi như "đã thấy ở chỉ số `-1`".
2. Duyệt `i, x`: cập nhật `s += 1 if x else -1`.
3. Nếu `s` đã có trong `first`: ứng viên độ dài là `i - first[s]`.
4. Nếu chưa có: lưu `first[s] = i`.

Chỉ số kiểu mảng: tiền tố sau khi đọc phần tử `i` ứng với "thời điểm `i + 1`", nên đoạn `(first[s], i]` có độ dài `i - first[s]`, **không cộng 1**.

In [12]:
def hashmap_first(nums):
    first = {0: -1}       # tổng 0 xuất hiện "tại chỉ số -1" (trước khi đọc gì)
    s = 0
    best = 0
    for i, x in enumerate(nums):
        s += 1 if x else -1
        if s in first:
            best = max(best, i - first[s])
        else:
            first[s] = i
    return best

for nums, expected in examples:
    got = hashmap_first(nums)
    print(nums, "->", got)
    assert got == expected

[0, 1] -> 2
[0, 1, 0] -> 2
[0, 1, 1, 1, 1, 1, 0, 0, 0] -> 6


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def findMaxLength(self, nums: List[int]) -> int:
        first = {0: -1}
        s = best = 0
        for i, x in enumerate(nums):
            s += 1 if x else -1
            if s in first:
                best = max(best, i - first[s])
            else:
                first[s] = i
        return best
```

**Độ phức tạp:** một lượt duyệt, mỗi bước thao tác dict `O(1)` trung bình: Time `O(n)`. Dict chứa tối đa `n + 1` khóa: Space `O(n)`.

### 7.3. Cách 4: Mảng offset thay cho dict

**Ý tưởng:** `s` luôn nằm trong khoảng `[-n, n]` (mỗi bước đổi đúng 1). Cộng thêm `n` để biến thành chỉ số không âm `[0, 2n]`, rồi dùng một **mảng** cỡ `2n + 1` thay cho dict. Giá trị `-2` nghĩa là "chưa thấy" (hoặc dùng `None`).

**Giải thích từng đoạn:**

1. `first = [None] * (2 * n + 1)`, `first[0 + n] = -1` (tiền tố rỗng).
2. Mỗi bước, `idx = s + n`. Nếu `first[idx]` là `None` thì ghi `i`, ngược lại cập nhật đáp án.

Ưu điểm: không băm, chỉ truy cập chỉ số, thường nhanh hơn dict trong thực tế một chút; nhược: dùng `2n + 1` ô dù ít khi cần hết.

In [13]:
def array_offset(nums):
    n = len(nums)
    first = [None] * (2 * n + 1)
    first[n] = -1              # s = 0 ứng với chỉ số 0 + n
    s = best = 0
    for i, x in enumerate(nums):
        s += 1 if x else -1
        idx = s + n            # dịch để không âm
        if first[idx] is None:
            first[idx] = i
        else:
            best = max(best, i - first[idx])
    return best

for nums, expected in examples:
    assert array_offset(nums) == expected
print("Cách 4 chạy đúng 3 ví dụ")

Cách 4 chạy đúng 3 ví dụ


**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def findMaxLength(self, nums: List[int]) -> int:
        n = len(nums)
        first = [None] * (2 * n + 1)
        first[n] = -1
        s = best = 0
        for i, x in enumerate(nums):
            s += 1 if x else -1
            if first[s + n] is None:
                first[s + n] = i
            else:
                best = max(best, i - first[s + n])
        return best
```

**Độ phức tạp:** Time `O(n)`, Space `O(n)` (mảng `2n + 1` ô).

### 7.4. Cách 5: Gom nhóm theo giá trị tiền tố, lấy vị trí đầu và cuối

**Ý tưởng khác:** thay vì cập nhật đáp án ngay trong lúc duyệt, ta làm **hai bước**: (a) dựng `pref` rồi, với mỗi giá trị, ghi nhận **vị trí đầu** và **vị trí cuối** xuất hiện; (b) đáp án là `max(last[v] - first[v])` trên mọi giá trị `v`.

Điều này đúng vì với mỗi giá trị `v`, cặp xa nhất luôn là (lần đầu, lần cuối). Cách này tách bạch "gom nhóm" và "tính đáp án", dễ mở rộng khi đề đổi (ví dụ cần đếm số lần).

In [14]:
def group_first_last(nums):
    pref = [0]
    for x in nums:
        pref.append(pref[-1] + (1 if x else -1))
    first, last = {}, {}
    for t, v in enumerate(pref):
        if v not in first:
            first[v] = t
        last[v] = t                 # luôn ghi đè -> giữ lần cuối
    return max(last[v] - first[v] for v in first)

for nums, expected in examples:
    assert group_first_last(nums) == expected
print("Cách 5 chạy đúng 3 ví dụ")
print("không có đoạn cân bằng:", group_first_last([1, 1, 1]))

Cách 5 chạy đúng 3 ví dụ
không có đoạn cân bằng: 0


**Code nộp LeetCode (Cách 5):**

```python
class Solution:
    def findMaxLength(self, nums: List[int]) -> int:
        first, last = {}, {}
        s = 0
        first[0] = last[0] = 0
        for t, x in enumerate(nums, 1):
            s += 1 if x else -1
            if s not in first:
                first[s] = t
            last[s] = t
        return max(last[v] - first[v] for v in first)
```

**Độ phức tạp:** Time `O(n)`, Space `O(n)` (hai dict). Hàm `max(...)` luôn có ít nhất khóa `0`, nên không bao giờ rỗng; nếu không có đoạn cân bằng thì mọi hiệu bằng `0` và đáp án là `0`.

## 8. TRACE từng bước

Ta mô phỏng Cách 3 với `nums = [0,1,1,0,0,0,1,1]` (đáp án `8`, vì cả mảng có 4 số `0` và 4 số `1`).

Bảng trạng thái (`first` ban đầu `{0: -1}`, `s = 0`, `best = 0`):

| `i` | `x` | `s` sau khi cộng | `s` đã có trong `first`? | `first` sau bước | `best` |
|---|---|---|---|---|---|
| 0 | 0 | -1 | Không | `{0:-1, -1:0}` | 0 |
| 1 | 1 | 0 | Có, `first[0] = -1` → độ dài `1 - (-1) = 2` | không đổi | 2 |
| 2 | 1 | 1 | Không | thêm `1:2` | 2 |
| 3 | 0 | 0 | Có → `3 - (-1) = 4` | không đổi | 4 |
| 4 | 0 | -1 | Có, `first[-1] = 0` → `4 - 0 = 4` | không đổi | 4 |
| 5 | 0 | -2 | Không | thêm `-2:5` | 4 |
| 6 | 1 | -1 | Có → `6 - 0 = 6` | không đổi | 6 |
| 7 | 1 | 0 | Có → `7 - (-1) = 8` | không đổi | 8 |

Chú ý hai điều: (1) khi `s = 0` ở `i = 1` ta dùng được nhờ gốc `{0: -1}`; (2) ở `i = 3`, `s = 0` lại gặp lại, nhưng **ta không ghi đè** `first[0]`, nên khoảng được đo từ `-1`, không từ `1`.

In [15]:
def trace_hashmap(nums):
    first = {0: -1}
    s = best = 0
    print("khởi tạo: first =", first, "| s =", s, "| best =", best)
    for i, x in enumerate(nums):
        s += 1 if x else -1
        print(f"\n[i={i}] x={x} -> s={s}")
        if s in first:
            cand = i - first[s]
            print(f"  s đã có, first[{s}] = {first[s]} -> độ dài {i} - ({first[s]}) = {cand}")
            if cand > best:
                best = cand
                print("  best tăng lên", best)
            else:
                print("  best giữ nguyên", best)
        else:
            first[s] = i
            print(f"  s mới -> first[{s}] = {i}")
        print("  first =", first)
    print("\nKẾT QUẢ:", best)
    return best

trace_hashmap([0, 1, 1, 0, 0, 0, 1, 1])

khởi tạo: first = {0: -1} | s = 0 | best = 0

[i=0] x=0 -> s=-1
  s mới -> first[-1] = 0
  first = {0: -1, -1: 0}

[i=1] x=1 -> s=0
  s đã có, first[0] = -1 -> độ dài 1 - (-1) = 2
  best tăng lên 2
  first = {0: -1, -1: 0}

[i=2] x=1 -> s=1
  s mới -> first[1] = 2
  first = {0: -1, -1: 0, 1: 2}

[i=3] x=0 -> s=0
  s đã có, first[0] = -1 -> độ dài 3 - (-1) = 4
  best tăng lên 4
  first = {0: -1, -1: 0, 1: 2}

[i=4] x=0 -> s=-1
  s đã có, first[-1] = 0 -> độ dài 4 - (0) = 4
  best giữ nguyên 4
  first = {0: -1, -1: 0, 1: 2}

[i=5] x=0 -> s=-2
  s mới -> first[-2] = 5
  first = {0: -1, -1: 0, 1: 2, -2: 5}

[i=6] x=1 -> s=-1
  s đã có, first[-1] = 0 -> độ dài 6 - (0) = 6
  best tăng lên 6
  first = {0: -1, -1: 0, 1: 2, -2: 5}

[i=7] x=1 -> s=0
  s đã có, first[0] = -1 -> độ dài 7 - (-1) = 8
  best tăng lên 8
  first = {0: -1, -1: 0, 1: 2, -2: 5}

KẾT QUẢ: 8


8

Thử thêm một ví dụ có đoạn cân bằng **nằm giữa**, không chạm đầu mảng: `[1, 1, 1, 0, 1, 0, 0, 0, 1]`... hãy xem bảng `first` thay đổi thế nào và đoạn dài nhất bắt đầu từ đâu.

In [16]:
trace_hashmap([1, 1, 1, 0, 1, 0, 0, 0, 1])

khởi tạo: first = {0: -1} | s = 0 | best = 0

[i=0] x=1 -> s=1
  s mới -> first[1] = 0
  first = {0: -1, 1: 0}

[i=1] x=1 -> s=2
  s mới -> first[2] = 1
  first = {0: -1, 1: 0, 2: 1}

[i=2] x=1 -> s=3
  s mới -> first[3] = 2
  first = {0: -1, 1: 0, 2: 1, 3: 2}

[i=3] x=0 -> s=2
  s đã có, first[2] = 1 -> độ dài 3 - (1) = 2
  best tăng lên 2
  first = {0: -1, 1: 0, 2: 1, 3: 2}

[i=4] x=1 -> s=3
  s đã có, first[3] = 2 -> độ dài 4 - (2) = 2
  best giữ nguyên 2
  first = {0: -1, 1: 0, 2: 1, 3: 2}

[i=5] x=0 -> s=2
  s đã có, first[2] = 1 -> độ dài 5 - (1) = 4
  best tăng lên 4
  first = {0: -1, 1: 0, 2: 1, 3: 2}

[i=6] x=0 -> s=1
  s đã có, first[1] = 0 -> độ dài 6 - (0) = 6
  best tăng lên 6
  first = {0: -1, 1: 0, 2: 1, 3: 2}

[i=7] x=0 -> s=0
  s đã có, first[0] = -1 -> độ dài 7 - (-1) = 8
  best tăng lên 8
  first = {0: -1, 1: 0, 2: 1, 3: 2}

[i=8] x=1 -> s=1
  s đã có, first[1] = 0 -> độ dài 8 - (0) = 8
  best giữ nguyên 8
  first = {0: -1, 1: 0, 2: 1, 3: 2}

KẾT QUẢ: 8


8

Quan sát: khi `s` quay lại một giá trị đã có thì ta mới "đo", và khoảng đo luôn tính từ **lần đầu** `s` xuất hiện. Nếu `s` không bao giờ lặp lại (ví dụ `[1,1,1]` có `s = 1,2,3`) thì `best` mãi bằng `0`.

## 9. Code nộp LeetCode

Ta nộp Cách 3 (hash map) vì ngắn gọn, `O(n)`, đúng ý tưởng nhất.

In [17]:
class Solution:
    def findMaxLength(self, nums: List[int]) -> int:
        first = {0: -1}          # tiền tố rỗng: tổng 0 tại chỉ số -1
        s = best = 0
        for i, x in enumerate(nums):
            s += 1 if x else -1  # 1 -> +1, 0 -> -1
            if s in first:
                best = max(best, i - first[s])
            else:
                first[s] = i     # chỉ lưu lần đầu
        return best


sol = Solution()
for nums, expected in examples:
    got = sol.findMaxLength(nums)
    print(nums, "->", got, "| kỳ vọng", expected)
    assert got == expected

[0, 1] -> 2 | kỳ vọng 2
[0, 1, 0] -> 2 | kỳ vọng 2
[0, 1, 1, 1, 1, 1, 0, 0, 0] -> 6 | kỳ vọng 6


## 10. Vì sao đúng

Gọi `P[t]` là tổng tiền tố sau `t` phần tử (với `1 -> +1`, `0 -> -1`, `P[0] = 0`).

**Điều 1: một đoạn cân bằng tương ứng với một cặp tiền tố bằng nhau.** Đoạn `nums[j:i]` có số `1` bằng số `0` ⇔ tổng các `±1` trong đoạn bằng `0` ⇔ `P[i] - P[j] = 0`.

**Điều 2: với mỗi `i`, đoạn dài nhất kết thúc tại `i` bắt đầu ở vị trí `j` nhỏ nhất thỏa `P[j] = P[i]`.** Vị trí đó chính là lần đầu `P` nhận giá trị `P[i]`. Thuật toán lưu đúng `first[v]` và không ghi đè, nên khi tới `i` nó biết ngay `j` tốt nhất.

**Điều 3: đáp án là max trên mọi `i`.** Mọi đoạn cân bằng đều kết thúc ở một `i` nào đó, nên lấy max theo `i` bao phủ mọi đoạn.

**Điều 4: gốc `{0: -1}`.** Đoạn bắt đầu từ đầu mảng ứng với `j = 0` (tiền tố rỗng, `P[0] = 0`). Trong chỉ số mảng kiểu "thời điểm `t` = sau chỉ số `t-1`", tiền tố rỗng nằm tại chỉ số `-1`, nên `first[0] = -1` và độ dài `i - (-1) = i + 1`.

Ta kiểm tra bằng code hai tính chất: (a) với mọi đoạn tìm được, số `0` thật sự bằng số `1`; (b) không có đoạn cân bằng nào dài hơn.

In [18]:
def find_best_segment(nums):
    """Trả về (độ dài, đầu, cuối_không_tính) của đoạn tìm được theo hash map."""
    first = {0: -1}
    s = 0
    best = (0, 0, 0)
    for i, x in enumerate(nums):
        s += 1 if x else -1
        if s in first:
            if i - first[s] > best[0]:
                best = (i - first[s], first[s] + 1, i + 1)
        else:
            first[s] = i
    return best

rng = random.Random(1)
for _ in range(300):
    n = rng.randint(1, 14)
    nums = [rng.randint(0, 1) for _ in range(n)]
    length, a, b = find_best_segment(nums)
    seg = nums[a:b]
    assert len(seg) == length
    assert seg.count(0) == seg.count(1)                       # (a) đoạn tìm được hợp lệ
    for lo in range(n):
        for hi in range(lo + 1, n + 1):
            part = nums[lo:hi]
            if part.count(0) == part.count(1):
                assert hi - lo <= length                      # (b) không có đoạn dài hơn
print("300 mảng ngẫu nhiên: đoạn hợp lệ và tối đa")

300 mảng ngẫu nhiên: đoạn hợp lệ và tối đa


## 11. Time/Space Complexity

Với `n = len(nums)`:

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn đếm cộng dồn | `O(n^2)` | `O(1)` |
| 2. Tiền tố + hai vòng lặp | `O(n^2)` | `O(n)` |
| 3. Hash map vị trí đầu | `O(n)` | `O(n)` |
| 4. Mảng offset | `O(n)` | `O(n)` |
| 5. Gom nhóm đầu/cuối | `O(n)` | `O(n)` |

**Time của Cách 3:** duyệt đúng `n` phần tử, mỗi bước chỉ cộng, tra `in`, đọc hoặc ghi dict, đều `O(1)` trung bình. **Space:** `s` chỉ nhận tối đa `n + 1` giá trị khác nhau (đoạn `[-n, n]` nhưng thực tế mỗi bước đổi 1 nên số giá trị phân biệt tối đa `n + 1`), nên dict có `O(n)` khóa.

## 12. Bộ test

Ta kiểm tra theo từng nhóm: ví dụ của đề, edge case, phản ví dụ cho các cách làm sai, đối chiếu tất cả cách giải với oracle độc lập trên nhiều input nhỏ ngẫu nhiên (seed cố định), và cuối cùng là input lớn đúng ràng buộc.

In [19]:
def oracle(nums):
    """Đáp án chuẩn: thử mọi lát cắt và dùng list.count, độc lập hoàn toàn với các cách giải."""
    n = len(nums)
    best = 0
    for lo in range(n):
        for hi in range(lo + 1, n + 1):
            seg = nums[lo:hi]
            if seg.count(0) == seg.count(1):
                best = max(best, hi - lo)
    return best

# 12.1 ví dụ của đề
for nums, expected in examples:
    assert Solution().findMaxLength(nums) == expected == oracle(nums)

# 12.2 edge case
edge = [
    ([0], 0),                       # một phần tử: không thể cân bằng
    ([1], 0),
    ([0, 0, 0, 0], 0),              # toàn 0
    ([1, 1, 1], 0),                 # toàn 1
    ([0, 1], 2),                    # đúng một cặp
    ([1, 0], 2),
    ([0, 1, 0, 1, 0, 1], 6),        # xen kẽ: cả mảng
    ([0, 0, 1, 1], 4),              # cả mảng, chia hai khối
    ([1, 0, 0, 0, 0], 2),           # đoạn cân bằng ở đầu
    ([0, 0, 0, 0, 1, 0], 2),        # đoạn cân bằng ở cuối
    ([0, 0, 1, 0, 0, 0, 1, 1], 6),  # cần bỏ đi phần tử đầu
]
for nums, expected in edge:
    assert Solution().findMaxLength(nums) == expected, nums
    assert oracle(nums) == expected, nums
print("ví dụ đề + edge case: OK")

ví dụ đề + edge case: OK


In [20]:
# 12.3 phản ví dụ: những cách làm SAI phổ biến, xem chúng sai ở đâu
def wrong_no_seed(nums):                 # quên gốc {0: -1}
    first = {}
    s = best = 0
    for i, x in enumerate(nums):
        s += 1 if x else -1
        if s in first:
            best = max(best, i - first[s])
        else:
            first[s] = i
    return best

def wrong_overwrite(nums):               # luôn ghi đè vị trí -> đo từ lần SAU CÙNG
    first = {0: -1}
    s = best = 0
    for i, x in enumerate(nums):
        s += 1 if x else -1
        if s in first:
            best = max(best, i - first[s])
        first[s] = i
    return best

def wrong_global_count(nums):            # so sánh đếm toàn mảng
    return len(nums) if nums.count(0) == nums.count(1) else 0

for nums in ([0, 1], [0, 1, 1, 0, 0, 1], [0, 1, 0, 0, 1, 1]):
    print(nums, "đúng =", oracle(nums),
          "| quên gốc =", wrong_no_seed(nums),
          "| ghi đè =", wrong_overwrite(nums),
          "| đếm toàn mảng =", wrong_global_count(nums))
assert wrong_no_seed([0, 1]) != oracle([0, 1])
assert wrong_overwrite([0, 1, 1, 0, 0, 1]) != oracle([0, 1, 1, 0, 0, 1])
assert wrong_global_count([1, 0, 0, 0, 0]) != oracle([1, 0, 0, 0, 0])
print("các cách sai bị phát hiện đúng như dự đoán")

[0, 1] đúng = 2 | quên gốc = 0 | ghi đè = 2 | đếm toàn mảng = 2
[0, 1, 1, 0, 0, 1] đúng = 6 | quên gốc = 4 | ghi đè = 4 | đếm toàn mảng = 6
[0, 1, 0, 0, 1, 1] đúng = 6 | quên gốc = 4 | ghi đè = 4 | đếm toàn mảng = 6
các cách sai bị phát hiện đúng như dự đoán


In [21]:
# 12.4 đối chiếu cả 5 cách với oracle trên input nhỏ ngẫu nhiên (seed cố định)
methods = {
    "brute_force": brute_force,
    "prefix_pairs": prefix_pairs,
    "hashmap_first": hashmap_first,
    "array_offset": array_offset,
    "group_first_last": group_first_last,
    "Solution": Solution().findMaxLength,
}
rng = random.Random(2024)
count = 0
for _ in range(1500):
    n = rng.randint(1, 16)
    p1 = rng.choice([0.2, 0.5, 0.8])          # thay đổi độ lệch giữa 0 và 1
    nums = [1 if rng.random() < p1 else 0 for _ in range(n)]
    expected = oracle(nums)
    for name, fn in methods.items():
        assert fn(nums[:]) == expected, (name, nums)
    count += 1
print("đối chiếu", count, "mảng ngẫu nhiên với", len(methods), "cách: OK")

đối chiếu 1500 mảng ngẫu nhiên với 6 cách: OK


In [22]:
# 12.5 input lớn đúng ràng buộc n = 10^5: chỉ chạy các cách O(n), hash/array/group phải cho cùng kết quả
rng = random.Random(7)
big_cases = {
    "ngẫu nhiên": [rng.randint(0, 1) for _ in range(10**5)],
    "toàn 0": [0] * 10**5,
    "xen kẽ": [i % 2 for i in range(10**5)],
    "nửa 0 nửa 1": [0] * 50000 + [1] * 50000,
    "lệch nhiều": [1 if rng.random() < 0.9 else 0 for _ in range(10**5)],
}
fast = ["hashmap_first", "array_offset", "group_first_last", "Solution"]
for title, arr in big_cases.items():
    t0 = time.perf_counter()
    results = {name: methods[name](arr) for name in fast}
    dt = time.perf_counter() - t0
    assert len(set(results.values())) == 1, results
    print(f"{title:14s} -> {results['Solution']:6d}   ({dt * 1000:.0f} ms cho 4 cách)")
assert methods["Solution"]([0] * 50000 + [1] * 50000) == 100000
assert methods["Solution"]([0] * 10**5) == 0
print("input lớn: OK")

ngẫu nhiên     ->  56884   (52 ms cho 4 cách)


toàn 0         ->      0   (66 ms cho 4 cách)


xen kẽ         -> 100000   (39 ms cho 4 cách)
nửa 0 nửa 1    -> 100000   (50 ms cho 4 cách)
lệch nhiều     ->     20   (49 ms cho 4 cách)


input lớn: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên "gốc" `{0: -1}`

Đoạn cân bằng bắt đầu **từ đầu mảng** sẽ bị bỏ sót, vì không có tiền tố nào trước nó để so sánh.

In [23]:
def buggy_1(nums):
    first = {}                      # SAI: thiếu {0: -1}
    s = best = 0
    for i, x in enumerate(nums):
        s += 1 if x else -1
        if s in first:
            best = max(best, i - first[s])
        else:
            first[s] = i
    return best

print("SAI   [0,1]  ->", buggy_1([0, 1]), "(đúng phải là 2)")
print("SAI   [0,0,1,1] ->", buggy_1([0, 0, 1, 1]), "(đúng phải là 4)")

SAI   [0,1]  -> 0 (đúng phải là 2)
SAI   [0,0,1,1] -> 2 (đúng phải là 4)


**Vì sao sai:** với `[0,1]`: `s` lần lượt `-1, 0`. `s = 0` chưa có trong `first` (vì thiếu gốc) nên chỉ ghi vào, không đo. **Sửa:** khởi tạo `first = {0: -1}`.

### Lỗi 2: Ghi đè vị trí cũ

Nếu mỗi lần gặp `s` ta đều ghi `first[s] = i`, thì khoảng đo bị **ngắn lại** vì điểm đầu bị dời về gần hơn.

In [24]:
def buggy_2(nums):
    first = {0: -1}
    s = best = 0
    for i, x in enumerate(nums):
        s += 1 if x else -1
        if s in first:
            best = max(best, i - first[s])
        first[s] = i                # SAI: luôn ghi đè
    return best

def fixed_2(nums):
    first = {0: -1}
    s = best = 0
    for i, x in enumerate(nums):
        s += 1 if x else -1
        if s in first:
            best = max(best, i - first[s])
        else:
            first[s] = i            # ĐÚNG: chỉ ghi lần đầu
    return best

nums = [0, 1, 1, 0, 0, 1]
print("SAI  :", buggy_2(nums))
print("ĐÚNG :", fixed_2(nums))
assert fixed_2(nums) == oracle(nums) != buggy_2(nums)

SAI  : 4
ĐÚNG : 6


**Vì sao sai:** khi `s = 0` gặp lại ở `i = 1`, ghi đè `first[0] = 1`. Về sau `s = 0` lại gặp ở `i = 5` nhưng chỉ đo `5 - 1 = 4` thay vì `5 - (-1) = 6`. **Sửa:** chỉ ghi `first[s]` khi chưa có (`else`).

### Lỗi 3: Chỉ so sánh tổng số `0` và `1` của cả mảng

Đề hỏi đoạn **con** dài nhất, không phải cả mảng. Cả mảng có thể không cân bằng nhưng đoạn con lại có.

In [25]:
def buggy_3(nums):
    return len(nums) if nums.count(0) == nums.count(1) else 0   # SAI

nums = [1, 0, 0, 0, 0]
print("SAI  :", buggy_3(nums))
print("ĐÚNG :", Solution().findMaxLength(nums), "(đoạn [1,0])")

SAI  : 0
ĐÚNG : 2 (đoạn [1,0])


**Vì sao sai:** mảng `[1,0,0,0,0]` có 1 số `1` và 4 số `0` nên bị trả `0`, nhưng đoạn `[1,0]` vẫn cân bằng. **Sửa:** dùng tiền tố, xét mọi cặp thời điểm.

### Lỗi 4: Nhầm độ dài `i - j + 1`

Ở dạng chỉ số mảng, `first[s]` là chỉ số **cuối của phần đã bị loại**, không phải điểm bắt đầu đoạn. Đoạn thật là chỉ số `first[s] + 1 .. i`, độ dài `i - first[s]`. Cộng thêm 1 sẽ **dư 1**.

In [26]:
def buggy_4(nums):
    first = {0: -1}
    s = best = 0
    for i, x in enumerate(nums):
        s += 1 if x else -1
        if s in first:
            best = max(best, i - first[s] + 1)    # SAI: dư 1
        else:
            first[s] = i
    return best

print("SAI  [0,1] ->", buggy_4([0, 1]), "(đúng là 2)")
print("ĐÚNG [0,1] ->", Solution().findMaxLength([0, 1]))

SAI  [0,1] -> 3 (đúng là 2)
ĐÚNG [0,1] -> 2


**Vì sao sai:** với `[0,1]`: tại `i = 1`, `s = 0`, `first[0] = -1`. `i - first[s] = 1 - (-1) = 2` đã đúng, cộng thêm 1 thành 3 (dài hơn cả mảng!). **Sửa:** bỏ `+ 1`. Kiểm tra nhanh: đáp án không thể vượt `len(nums)`.

## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận ra bài dạng này:**

- Đề hỏi đoạn con **liên tiếp** với điều kiện "**bằng nhau**" giữa hai loại phần tử (số 0 và số 1, số chẵn và số lẻ, chữ hoa và chữ thường...).
- Có thể biến điều kiện thành "**tổng bằng 0**" bằng cách gán trọng số `+1` / `-1`.
- Hỏi đoạn **dài nhất** (lưu vị trí đầu tiên) hoặc **ngắn nhất** (lưu vị trí gần nhất, ghi đè) hoặc **đếm số đoạn** (đếm tần suất của tiền tố).

**Mẹo:**

1. "Tổng đoạn = hiệu hai tiền tố": hầu hết bài "đoạn con có tổng = k" giải bằng `pref[i] - k` tra trong dict.
2. Luôn thêm tiền tố rỗng `0` vào đầu (hash map `{0: -1}` hoặc `{0: 1}` khi đếm).
3. Tìm đoạn **dài nhất** thì chỉ lưu **lần đầu**; tìm đoạn **ngắn nhất** thì ghi đè để lấy **lần gần nhất**.
4. Giá trị tiền tố nằm trong `[-n, n]` nên có thể thay dict bằng mảng offset khi cần tốc độ.
5. Sau khi làm xong, hỏi lại: "Có cần gốc? Có thể âm? Độ dài `i - j` hay `i - j + 1`?"

## 15. Biến thể

### 15.1. Trả về chính đoạn con (không chỉ độ dài)

Chỉ cần nhớ thêm vị trí bắt đầu của đoạn tốt nhất. Nếu có nhiều đoạn cùng độ dài, ta lấy đoạn **xuất hiện sớm nhất** (do dùng `>` chặt).

In [27]:
def longest_balanced_subarray(nums):
    first = {0: -1}
    s = 0
    best_len, best_start = 0, 0
    for i, x in enumerate(nums):
        s += 1 if x else -1
        if s in first:
            if i - first[s] > best_len:
                best_len, best_start = i - first[s], first[s] + 1
        else:
            first[s] = i
    return nums[best_start:best_start + best_len]

print(longest_balanced_subarray([0, 1, 1, 1, 1, 1, 0, 0, 0]))   # [1,1,1,0,0,0]
print(longest_balanced_subarray([0, 1, 0]))                      # [0,1]
print(longest_balanced_subarray([1, 1]))                         # []

[1, 1, 1, 0, 0, 0]
[0, 1]
[]


### 15.2. Đếm **số** đoạn con cân bằng

Nếu hai tiền tố bằng nhau thì có một đoạn. Nếu giá trị `v` xuất hiện `c` lần (tính cả tiền tố rỗng) thì có `C(c, 2) = c(c-1)/2` đoạn. Có thể đếm trực tiếp trong một lượt: mỗi lần gặp `s`, cộng thêm "số lần đã thấy `s` trước đó".

In [28]:
from collections import Counter

def count_balanced(nums):
    seen = Counter({0: 1})        # tiền tố rỗng đã thấy 1 lần
    s = total = 0
    for x in nums:
        s += 1 if x else -1
        total += seen[s]          # mỗi lần thấy trước đó = một đoạn kết thúc ở đây
        seen[s] += 1
    return total

def count_balanced_brute(nums):
    n = len(nums)
    return sum(1 for lo in range(n) for hi in range(lo + 1, n + 1)
               if nums[lo:hi].count(0) == nums[lo:hi].count(1))

print(count_balanced([0, 1, 0]))       # [0,1], [1,0] -> 2
rng = random.Random(5)
for _ in range(200):
    arr = [rng.randint(0, 1) for _ in range(rng.randint(1, 12))]
    assert count_balanced(arr) == count_balanced_brute(arr)
print("count_balanced khớp vét cạn")

2
count_balanced khớp vét cạn


### 15.3. Các bài cùng họ

- **LeetCode 560 — Subarray Sum Equals K:** đoạn có tổng bằng `k`, đếm số đoạn. Tương tự 15.2 nhưng tra `s - k`.
- **LeetCode 523 — Continuous Subarray Sum:** tổng chia hết cho `k`; lưu **`s % k`** vị trí đầu tiên.
- **LeetCode 974 — Subarray Sums Divisible by K:** đếm số đoạn có tổng chia hết cho `k` bằng `Counter` của `s % k`.
- **Biến thể ba loại (0/1/2 bằng nhau):** thay `s` bằng cặp `(count1 - count0, count2 - count1)`, dùng làm khóa dict.
- **Ma trận nhị phân:** ép từng hàng/cột thành bài mảng này (ví dụ tìm hình chữ nhật có số 0 và 1 bằng nhau) bằng tiền tố 2 chiều.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn đếm cộng dồn | thử mọi cặp (đầu, cuối), đếm 0 và 1 | `O(n^2)` | `O(1)` | `n` rất nhỏ, dùng làm oracle để kiểm tra |
| 2. Tiền tố + hai vòng | đổi sang `pref`, tìm hai tiền tố bằng nhau xa nhất | `O(n^2)` | `O(n)` | bước đệm để hiểu ý tưởng tiền tố |
| 3. Hash map vị trí đầu | vừa duyệt vừa lưu `first[s]` | `O(n)` | `O(n)` | **cách nộp chuẩn**, ngắn và rõ |
| 4. Mảng offset | thay dict bằng mảng `2n + 1` ô, chỉ số = `s + n` | `O(n)` | `O(n)` | cần tốc độ, biết trước miền giá trị |
| 5. Gom nhóm đầu/cuối | dựng `pref`, mỗi giá trị lấy `last - first` | `O(n)` | `O(n)` | dễ mở rộng (đếm, liệt kê); hai lượt duyệt |

Kết luận: ba cách cuối đều tuyến tính. Cách 3 là lựa chọn mặc định khi thi hoặc phỏng vấn.

## 17. Bài tập vận dụng

Làm lần lượt từ dễ đến khó. Đáp án nằm ở **mục 19**. Các cell TODO bên dưới **chưa được gọi**, bạn tự điền rồi tự chạy.

### Bài 1 (đoán output)

Không chạy code, hãy đoán `s` ở cuối và giá trị trả về:

```python
nums = [1, 1, 0, 0, 1]
first = {0: -1}
s = best = 0
for i, x in enumerate(nums):
    s += 1 if x else -1
    if s in first:
        best = max(best, i - first[s])
    else:
        first[s] = i
print(s, best)
```

### Bài 2 (tính tay)

Với `nums = [0, 0, 1, 0, 0, 0, 1, 1]`, hãy viết dãy `pref` (9 phần tử, bắt đầu từ `0`) rồi tìm cặp tiền tố bằng nhau xa nhất. Đáp án là bao nhiêu?

### Bài 3 (điền TODO)

Điền chỗ trống để hàm trả về độ dài đoạn cân bằng dài nhất:

In [29]:
def ex3_fill(nums):
    first = {0: -1}
    s = best = 0
    for i, x in enumerate(nums):
        s += ...            # TODO 1: cộng bao nhiêu khi x = 1, x = 0?
        if s in first:
            best = ...      # TODO 2: cập nhật best bằng độ dài đoạn
        else:
            ...             # TODO 3: lưu vị trí đầu tiên của s
    return best

# Khi đã điền xong, bỏ dấu # ở dòng sau để thử:
# assert ex3_fill([0, 1, 0]) == 2

### Bài 4 (tự viết lại)

Viết lại **Cách 4 (mảng offset)** từ đầu mà không nhìn bài giảng. Gợi ý khung: tạo mảng `2n + 1` ô, đặt `first[n] = -1`, duyệt và cập nhật như Cách 3.

In [30]:
def ex4_array_offset(nums):
    # TODO: tự viết toàn bộ hàm (mảng offset thay cho dict)
    ...

# Khi đã viết xong, bỏ dấu # để thử:
# assert ex4_array_offset([0, 1, 1, 1, 1, 1, 0, 0, 0]) == 6

### Bài 5 (mini challenge, biến thể)

Viết hàm `shortest_balanced(nums)` trả về **độ dài đoạn cân bằng NGẮN nhất** (khác rỗng), hoặc `0` nếu không có. Gợi ý: để tìm đoạn ngắn nhất, ta cần vị trí **gần nhất** của mỗi tiền tố, tức là phải ghi đè mỗi lần. Thử nghĩ: đoạn ngắn nhất luôn có độ dài mấy trong mảng nhị phân? (Câu trả lời thú vị!)

In [31]:
def ex5_shortest_balanced(nums):
    # TODO: trả về độ dài đoạn cân bằng ngắn nhất, 0 nếu không có
    ...

# Khi đã viết xong, bỏ dấu # để thử:
# assert ex5_shortest_balanced([0, 0, 1, 1]) == 2

## 18. Gợi ý 3 mức

**Bài 3**

- Mức 1 (nhẹ): `s` thể hiện "số `1` trừ số `0`" cho tới nay, nên `x = 1` làm `s` thay đổi một cách, `x = 0` làm thay đổi cách ngược lại.
- Mức 2 (vừa): TODO 2 dùng `max(best, ...)` với độ dài tính từ `first[s]`; TODO 3 gán `first[s]` bằng chỉ số hiện tại.
- Mức 3 (gần đáp án): `s += 1 if x else -1`; `best = max(best, i - first[s])`; `first[s] = i`.

**Bài 4**

- Mức 1: `s` nằm trong `[-n, n]`, vậy chỉ số `s + n` nằm trong `[0, 2n]`.
- Mức 2: ô trống chứa `None`; chỉ ghi `first[idx] = i` khi `first[idx] is None`.
- Mức 3: đặt `first[n] = -1` trước khi duyệt.

**Bài 5**

- Mức 1: mỗi lần đọc một phần tử, `s` đổi đúng 1 đơn vị. Có đoạn độ dài 1 cân bằng không?
- Mức 2: đoạn cân bằng có số `0` bằng số `1` nên độ dài luôn **chẵn**, ngắn nhất là 2.
- Mức 3: đoạn độ dài 2 cân bằng chỉ khi hai phần tử kề nhau **khác nhau**, vậy chỉ cần tìm một cặp kề nhau khác nhau. Có thể mở rộng: đoạn ngắn nhất tổng quát (giữ nguyên ý ghi đè) vẫn cho kết quả 2.

## 19. Đáp án tham khảo

**Bài 1:** `nums = [1,1,0,0,1]` cho `s` lần lượt `1, 2, 1, 0, 1`. `s = 1` ở `i = 2` đã có (`first[1] = 0`) nên độ dài `2`; `s = 0` ở `i = 3` đã có (`first[0] = -1`) nên độ dài `4`; `s = 1` ở `i = 4` có `first[1] = 0` nên độ dài `4`. In ra `1 4`.

**Bài 2:** `pref = [0, -1, -2, -1, -2, -3, -4, -3, -2]`. Giá trị `-2` xuất hiện ở thời điểm `2, 4, 8`; xa nhất là `8 - 2 = 6`. Giá trị `-1` ở `1, 3`: `2`. Giá trị `-3` ở `5, 7`: `2`. Đáp án `6`.

Các bài 3, 4, 5 có code chạy kiểm tra bằng `assert` ngay bên dưới.

In [32]:
# Bài 1: kiểm tra bằng cách chạy
nums = [1, 1, 0, 0, 1]
first = {0: -1}
s = best = 0
for i, x in enumerate(nums):
    s += 1 if x else -1
    if s in first:
        best = max(best, i - first[s])
    else:
        first[s] = i
assert (s, best) == (1, 4)
print("Bài 1:", s, best)

# Bài 2
nums = [0, 0, 1, 0, 0, 0, 1, 1]
pref = [0]
for x in nums:
    pref.append(pref[-1] + (1 if x else -1))
assert pref == [0, -1, -2, -1, -2, -3, -4, -3, -2]
assert Solution().findMaxLength(nums) == 6 == oracle(nums)
print("Bài 2:", pref, "-> 6")

# Bài 3
def ex3_answer(nums):
    first = {0: -1}
    s = best = 0
    for i, x in enumerate(nums):
        s += 1 if x else -1
        if s in first:
            best = max(best, i - first[s])
        else:
            first[s] = i
    return best

# Bài 4
def ex4_answer(nums):
    n = len(nums)
    first = [None] * (2 * n + 1)
    first[n] = -1
    s = best = 0
    for i, x in enumerate(nums):
        s += 1 if x else -1
        if first[s + n] is None:
            first[s + n] = i
        else:
            best = max(best, i - first[s + n])
    return best

# Bài 5: đoạn cân bằng ngắn nhất (độ dài chẵn, ngắn nhất là 2 nếu có hai phần tử kề khác nhau)
def ex5_answer(nums):
    first_zero_len = 0
    last = {0: -1}
    s = 0
    best = 0
    for i, x in enumerate(nums):
        s += 1 if x else -1
        if s in last:
            cand = i - last[s]
            if best == 0 or cand < best:
                best = cand
        last[s] = i            # ghi đè: lấy vị trí GẦN NHẤT
    return best

def ex5_oracle(nums):
    n = len(nums)
    ans = 0
    for lo in range(n):
        for hi in range(lo + 1, n + 1):
            seg = nums[lo:hi]
            if seg.count(0) == seg.count(1) and (ans == 0 or hi - lo < ans):
                ans = hi - lo
    return ans

assert ex3_answer([0, 1, 0]) == 2
assert ex4_answer([0, 1, 1, 1, 1, 1, 0, 0, 0]) == 6
assert ex5_answer([0, 0, 1, 1]) == 2
assert ex5_answer([1, 1, 1]) == 0
rng = random.Random(11)
for _ in range(500):
    arr = [rng.randint(0, 1) for _ in range(rng.randint(1, 14))]
    assert ex3_answer(arr) == ex4_answer(arr) == oracle(arr)
    assert ex5_answer(arr) == ex5_oracle(arr)
    # quan sát của bài 5: đáp án chỉ có thể là 0 hoặc 2
    assert ex5_answer(arr) in (0, 2)
print("Bài 3, 4, 5: đều khớp vét cạn trên 500 mảng ngẫu nhiên")

Bài 1: 1 4
Bài 2: [0, -1, -2, -1, -2, -3, -4, -3, -2] -> 6
Bài 3, 4, 5: đều khớp vét cạn trên 500 mảng ngẫu nhiên


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "độ dài đoạn con dài nhất có tổng bằng 0 sau khi gán trọng số":**

```python
class Solution:
    def findMaxLength(self, nums: List[int]) -> int:
        first = {0: -1}               # gốc: tiền tố rỗng tại chỉ số -1
        s = best = 0
        for i, x in enumerate(nums):
            s += 1 if x else -1       # 1 -> +1, 0 -> -1
            if s in first:
                best = max(best, i - first[s])   # đo từ LẦN ĐẦU
            else:
                first[s] = i                      # chỉ ghi lần đầu
        return best
```

**Checklist trước khi submit:**

- [ ] Đã khởi tạo `first = {0: -1}`?
- [ ] Chỉ ghi `first[s]` khi **chưa có** (không ghi đè)?
- [ ] Độ dài là `i - first[s]`, **không** cộng 1?
- [ ] `0` được tính là `-1` (không phải `0`)?
- [ ] Đã thử `[0]`, `[1]`, `[0,1]`, toàn `0`, toàn `1`, xen kẽ?
- [ ] Không có đoạn cân bằng thì trả `0`?
- [ ] Độ phức tạp `O(n)` Time, `O(n)` Space, phù hợp với `n = 10^5`?

**Ghi nhớ một câu:** *"Số 0 bằng số 1 ⇔ tổng ±1 bằng 0 ⇔ hai tiền tố bằng nhau; lấy lần xuất hiện đầu tiên để đoạn dài nhất."*